# Image Classification using Deep Learning (CNN)

**Objective:** Implement a deep learning CNN model using PyTorch for handwritten digit classification.

**Dataset:** sklearn Digits Dataset (8x8 images)

**Deliverables:**
- Functional CNN Model
- Training Loss Plot
- Test Accuracy
- Prediction Visualization


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split


In [ ]:
# Load dataset
digits = load_digits()
X = digits.images / 16.0
y = digits.target

# Add channel dimension (N, 1, 8, 8)
X = X[:, None, :, :]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


In [ ]:
# DataLoaders
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64
)


In [ ]:
# CNN Model
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.fc1 = nn.Linear(32 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = CNN()
print(model)


In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Training loop
losses = []

for epoch in range(10):
    model.train()
    epoch_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/10 - Loss: {avg_loss:.4f}")


In [ ]:
# Plot training loss
plt.figure()
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()


In [ ]:
# Evaluate model
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb).argmax(1)
        total += yb.size(0)
        correct += (preds == yb).sum().item()

accuracy = correct / total
print("Test Accuracy:", accuracy)


In [ ]:
# Visualize predictions
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i][0], cmap='gray')
    pred = model(X_test[i].unsqueeze(0)).argmax().item()
    plt.title(f"Pred: {pred}")
    plt.axis("off")

plt.show()
